In [1]:
print("""
@File         : Imputing categorical variables.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-25 16:25:43
@Email        : cuixuanstephen@gmail.com
@Description  : 插补分类变量
""")


@File         : Imputing categorical variables.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-25 16:25:43
@Email        : cuixuanstephen@gmail.com
@Description  : 插补分类变量



In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from feature_engine.imputation import CategoricalImputer

In [3]:
data = pd.read_csv('../../DATA/credit_approval_uci.csv')
X_train, X_test, y_train, y_test = train_test_split(
    data.drop("target", axis='columns'),
    data["target"], test_size=0.3,
    random_state=0,
)

In [4]:
categorical_vars = X_train.select_dtypes(include='O').columns.to_list()
categorical_vars

['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']

In [5]:
frequent_values = X_train[categorical_vars].mode().iloc[0].to_dict()
frequent_values

{'A1': 'b',
 'A4': 'u',
 'A5': 'g',
 'A6': 'c',
 'A7': 'v',
 'A9': 't',
 'A10': 'f',
 'A12': 'f',
 'A13': 'g'}

In [6]:
X_train_t = X_train.fillna(value=frequent_values)
X_test_t = X_test.fillna(value=frequent_values)

为了用特定字符串替换缺失数据，让我们创建一个插补词典，以分类变量名称作为键，以任意字符串作为值：

In [7]:
imputation_dict = {var: 'no_data' for var in categorical_vars}

In [8]:
imputer = SimpleImputer(strategy='most_frequent')

In [12]:
ct = ColumnTransformer(
    transformers=[('imputer', imputer, categorical_vars)],
    remainder='passthrough', force_int_remainder_cols=False
).set_output(transform='pandas')

> 要使用字符串而不是最常见的类别来填补缺失数据，请按如下方式设置 SimpleImputer()：`imputer = SimpleImputer(strategy="constant", fill_value="missing")`。

In [13]:
ct.fit(X_train)

ColumnTransformer(force_int_remainder_cols=False, remainder='passthrough',
                  transformers=[('imputer',
                                 SimpleImputer(strategy='most_frequent'),
                                 ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10',
                                  'A12', 'A13'])])

In [14]:
ct.named_transformers_.imputer.statistics_

array(['b', 'u', 'g', 'c', 'v', 't', 'f', 'f', 'g'], dtype=object)

In [15]:
X_train_t = ct.transform(X_train)
X_test_t = ct.transform(X_test)

In [16]:
X_train_t.head()

,imputer__A1,imputer__A4,imputer__A5,imputer__A6,imputer__A7,imputer__A9,imputer__A10,imputer__A12,imputer__A13,remainder__A2,remainder__A3,remainder__A8,remainder__A11,remainder__A14,remainder__A15
596,a,u,g,c,v,t,t,t,g,46.08,3.000,2.375,8,396.0,4159
303,a,u,g,q,v,t,f,f,g,15.92,NaN,NaN,0,120.0,0
204,b,y,p,w,v,t,t,f,g,36.33,2.125,0.085,1,50.0,1187
351,b,y,p,ff,ff,t,f,f,g,22.17,NaN,NaN,0,100.0,0
118,b,u,g,m,v,t,t,t,g,57.83,7.040,14.000,6,360.0,1332


In [17]:
imputer = CategoricalImputer(imputation_method='frequent', variables=categorical_vars)

In [18]:
imputer.fit(X_train)

CategoricalImputer(imputation_method='frequent',
                   variables=['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12',
                              'A13'])

In [19]:
imputer.imputer_dict_

{'A1': 'b',
 'A4': 'u',
 'A5': 'g',
 'A6': 'c',
 'A7': 'v',
 'A9': 't',
 'A10': 'f',
 'A12': 'f',
 'A13': 'g'}

In [20]:
X_train_t = imputer.transform(X_train)
X_test_t = imputer.transform(X_test)

如果想使用 `CategoricalImputer()` 用字符串或最常见的值来估算数值变量，请将 `ignore_format` 参数设置为 `True`。这主要是为了有些情况下，数字表示分类变量。